## Hybrid Search

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

pinecone_api = os.getenv('PINECONE_API_KEY')

In [2]:
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')
os.environ['LNAGCHAIN_TRACING_V2'] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Hybrid Search"

In [3]:
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone, ServerlessSpec
index_name = "hybridsearch-langchain"

## Initialize the Pinecone Client
pc = Pinecone(api_key= pinecone_api)

In [4]:
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric='dotproduct',    ## For sparse values
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )

In [5]:
index = pc.Index(index_name)
index

c:\Users\singh\OneDrive\Desktop\Generative AI\GenAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
## Vector Embedding and Sparse Matrix
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embedding

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8874.37it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [7]:
## TF-IDF technique for sparse matrix
from pinecone_text.sparse import BM25Encoder
bm25 = BM25Encoder().default()
bm25

In [ ]:
senetences = [
    'In 2023, I visited Paris',
    'In 2022, I visited New York',
    'In 2021, I visited New Orleans'
]

bm25.fit(senetences)

## Store the value in json file
bm25.dump('bm25_values.json')

100%|██████████| 3/3 [00:00<00:00, 39.19it/s]


In [9]:
bm25_encoder = BM25Encoder().load("bm25_values.json")

In [10]:
retriever = PineconeHybridSearchRetriever(embeddings= embedding, sparse_encoder=bm25_encoder, index=index)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001FD6EEA6250>, index=<pinecone.db_data.index.Index object at 0x000001FD6FFE7F50>)

In [11]:
retriever.add_texts([
    'In 2023, I visited Paris',
    'In 2022, I visited New York',
    'In 2021, I visited New Orleans'
])

100%|██████████| 1/1 [00:05<00:00,  5.56s/it]


In [14]:
retriever.invoke("What City did I visit first?")

[Document(metadata={'score': 0.257887483}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.244532719}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.220131129}, page_content='In 2023, I visited Paris')]